In [0]:

# PHASE 5 — STEP 1
# Verify the existing Silver tables

catalog_name = "dbwteam3"
schema_name = "order_fulfillment"

silver_schema = f"{catalog_name}.{schema_name}"

# 1. List tables in the project schema
print("Tables available in the project schema:")
display(spark.sql(f"SHOW TABLES IN {silver_schema}"))

# 2. Identify Silver tables
tables_df = spark.sql(f"SHOW TABLES IN {silver_schema}")
table_names = [
    row.tableName
    for row in tables_df.collect()
]

silver_tables = [
    name for name in table_names
    if name.lower().startswith("silver")
    or "silver" in name.lower()
]

print("Silver tables identified:", silver_tables)

# 3. Inspect each Silver table's schema and row count
for table_name in silver_tables:
    full_table_name = f"{silver_schema}.{table_name}"

    print(f"\n--- {full_table_name} ---")

    df = spark.table(full_table_name)

    print("Schema:")
    df.printSchema()

    print("Row count:", df.count())


Tables available in the project schema:


database,tableName,isTemporary
order_fulfillment,bronze_delivery_events,false
order_fulfillment,bronze_order_lines,false
order_fulfillment,bronze_orders,false
order_fulfillment,bronze_pharmacies,false
order_fulfillment,bronze_schema_test,false
order_fulfillment,bronze_shipment_lines,false
order_fulfillment,order_line_fulfilment,false
order_fulfillment,order_service_performance,false
order_fulfillment,regional_backlog,false
order_fulfillment,silver_delivery_events,false


Silver tables identified: ['silver_delivery_events', 'silver_duplicate_delivery_events', 'silver_order_lines', 'silver_orders', 'silver_pharmacies', 'silver_quarantined_delivery_events', 'silver_rejected_delivery_events', 'silver_rejected_order_lines', 'silver_rejected_orders', 'silver_shipment_lines']

--- dbwteam3.order_fulfillment.silver_delivery_events ---
Schema:
root
 |-- event_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- line_id: string (nullable = true)
 |-- delivered_quantity: long (nullable = true)
 |-- delivery_time: timestamp (nullable = true)
 |-- is_overdelivery: boolean (nullable = true)
 |-- is_delivery_before_order: boolean (nullable = true)
 |-- total_delivered_quantity: long (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)

Row count: 6018

--- dbwteam3.order_fulfillment.silver_duplicate_d

In [0]:

# PHASE 5 — STEP 1B
# Inspect accepted Silver tables

catalog_name = "dbwteam3"
schema_name = "order_fulfillment"

silver_tables = [
    "silver_orders",
    "silver_order_lines",
    "silver_shipment_lines",
    "silver_delivery_events",
    "silver_pharmacies"
]

for table_name in silver_tables:
    full_name = f"{catalog_name}.{schema_name}.{table_name}"

    print(f"\n{'=' * 60}")
    print(f"TABLE: {full_name}")

    df = spark.table(full_name)

    print("Columns:", df.columns)
    print("Row count:", df.count())
    df.printSchema()



TABLE: dbwteam3.order_fulfillment.silver_orders
Columns: ['order_id', 'pharmacy_id', 'order_time', 'promised_delivery_time', 'order_status', '_source_file', '_ingestion_timestamp', '_batch_id']
Row count: 2040
root
 |-- order_id: string (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- promised_delivery_time: timestamp (nullable = true)
 |-- order_status: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)


TABLE: dbwteam3.order_fulfillment.silver_order_lines
Columns: ['order_id', 'line_id', 'product_id', 'ordered_quantity', '_source_file', '_ingestion_timestamp', '_batch_id']
Row count: 4773
root
 |-- order_id: string (nullable = true)
 |-- line_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- ordered_quantity: long (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _in

In [0]:

# PHASE 5 — STEP 1C
# Verify delivery-event uniqueness and quality flags

catalog_name = "dbwteam3"
schema_name = "order_fulfillment"

events_table = (
    f"{catalog_name}.{schema_name}.silver_delivery_events"
)

events = spark.table(events_table)

# 1. Total rows versus unique event IDs
events.selectExpr(
    "COUNT(*) AS total_rows",
    "COUNT(DISTINCT event_id) AS unique_event_ids",
    "COUNT_IF(event_id IS NULL) AS null_event_ids"
).show()

# 2. Check duplicate event IDs
duplicate_events = (
    events.groupBy("event_id")
    .count()
    .filter("count > 1")
)

print("Duplicate event IDs:", duplicate_events.count())
display(duplicate_events.limit(10))

# 3. Check existing exception flags
events.groupBy(
    "is_overdelivery",
    "is_delivery_before_order"
).count().orderBy(
    "is_overdelivery",
    "is_delivery_before_order"
).show()


+----------+----------------+--------------+
|total_rows|unique_event_ids|null_event_ids|
+----------+----------------+--------------+
|      6018|            6018|             0|
+----------+----------------+--------------+

Duplicate event IDs: 0


event_id,count


+---------------+------------------------+-----+
|is_overdelivery|is_delivery_before_order|count|
+---------------+------------------------+-----+
|          false|                   false| 5823|
|           true|                   false|  135|
|           true|                    true|   60|
+---------------+------------------------+-----+



In [0]:

# PHASE 5 — STEP 2
# Build order_line_fulfilment

from pyspark.sql import functions as F

catalog_name = "dbwteam3"
schema_name = "order_fulfillment"

orders_table = f"{catalog_name}.{schema_name}.silver_orders"
lines_table = f"{catalog_name}.{schema_name}.silver_order_lines"
events_table = f"{catalog_name}.{schema_name}.silver_delivery_events"

gold_table = f"{catalog_name}.{schema_name}.order_line_fulfilment"

# 1. Load accepted Silver records
orders = spark.table(orders_table)
order_lines = spark.table(lines_table)
events = spark.table(events_table)

# 2. Aggregate delivery events to one row per order line
# Keep uncapped delivered units for overdelivery checks.
delivery_summary = (
    events
    .groupBy("order_id", "line_id")
    .agg(
        F.sum("delivered_quantity").alias("uncapped_delivered_units"),
        F.max("delivery_time").alias("last_delivery_time"),
        F.max(
            F.when(
                F.col("delivered_quantity") > 0,
                F.col("delivery_time")
            )
        ).alias("last_positive_delivery_time"),
        F.sum(
            F.when(
                F.col("is_overdelivery") == True,
                F.col("delivered_quantity")
            ).otherwise(0)
        ).alias("flagged_overdelivery_units")
    )
)

# 3. Join order lines to aggregated deliveries
line_metrics = (
    order_lines
    .join(
        delivery_summary,
        ["order_id", "line_id"],
        "left"
    )
    .join(
        orders.select(
            "order_id",
            "order_time",
            "promised_delivery_time",
            "order_status"
        ),
        "order_id",
        "left"
    )
    .withColumn(
        "uncapped_delivered_units",
        F.coalesce(F.col("uncapped_delivered_units"), F.lit(0))
    )
    .withColumn(
        "capped_delivered_units",
        F.least(
            F.col("uncapped_delivered_units"),
            F.col("ordered_quantity")
        )
    )
    .withColumn(
        "open_units",
        F.greatest(
            F.lit(0),
            F.col("ordered_quantity") - F.col("capped_delivered_units")
        )
    )
    .withColumn(
        "fulfilment_percentage",
        F.when(
            F.col("ordered_quantity") > 0,
            F.round(
                F.col("capped_delivered_units") /
                F.col("ordered_quantity") * 100,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "is_overdelivered",
        F.col("uncapped_delivered_units") >
        F.col("ordered_quantity")
    )
    .withColumn(
        "completion_time",
        F.when(
            F.col("open_units") == 0,
            F.col("last_positive_delivery_time")
        )
    )
)

# 4. Write the Gold table
(
    line_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print("Gold table created:", gold_table)
print("Gold row count:", spark.table(gold_table).count())

display(
    spark.table(gold_table)
    .select(
        "order_id",
        "line_id",
        "ordered_quantity",
        "uncapped_delivered_units",
        "capped_delivered_units",
        "fulfilment_percentage",
        "open_units",
        "completion_time",
        "is_overdelivered"
    )
    .limit(20)
)


Gold table created: dbwteam3.order_fulfillment.order_line_fulfilment
Gold row count: 4773


order_id,line_id,ordered_quantity,uncapped_delivered_units,capped_delivered_units,fulfilment_percentage,open_units,completion_time,is_overdelivered
ORD-000001,L01,40,40,40,100.0,0,2026-08-02T03:45:30.000Z,false
ORD-000001,L02,20,20,20,100.0,0,2026-08-02T03:45:30.000Z,false
ORD-000002,L01,50,50,50,100.0,0,2026-08-04T13:25:56.000Z,false
ORD-000002,L02,60,60,60,100.0,0,2026-08-04T13:25:56.000Z,false
ORD-000003,L01,10,7,7,70.0,3,null,false
ORD-000003,L02,100,33,33,33.0,67,null,false
ORD-000003,L03,40,10,10,25.0,30,null,false
ORD-000003,L04,100,80,80,80.0,20,null,false
ORD-000003,L05,10,1,1,10.0,9,null,false
ORD-000003,L06,200,175,175,87.5,25,null,false


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.table(
    "dbwteam3.order_fulfillment.silver_delivery_events"
)

lines = spark.table(
    "dbwteam3.order_fulfillment.silver_order_lines"
)

# Calculate cumulative delivered units in event-time order
delivery_window = (
    Window.partitionBy("order_id", "line_id")
    .orderBy("delivery_time", "event_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

event_progress = events.withColumn(
    "cumulative_delivered_units",
    F.sum("delivered_quantity").over(delivery_window)
)

# First timestamp when cumulative delivery reaches ordered quantity
completion_times = (
    event_progress
    .join(
        lines.select("order_id", "line_id", "ordered_quantity"),
        ["order_id", "line_id"],
        "inner"
    )
    .filter(
        F.col("cumulative_delivered_units") >=
        F.col("ordered_quantity")
    )
    .groupBy("order_id", "line_id")
    .agg(
        F.min("delivery_time").alias("completion_time")
    )
)

# Update the existing Gold table without changing its other metrics
gold = spark.table(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)

corrected_gold = (
    gold.drop("completion_time")
    .join(
        completion_times,
        ["order_id", "line_id"],
        "left"
    )
)

(
    corrected_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "dbwteam3.order_fulfillment.order_line_fulfilment"
    )
)

print("Completion-time correction applied.")
print(
    "Gold row count:",
    spark.table(
        "dbwteam3.order_fulfillment.order_line_fulfilment"
    ).count()
)


Completion-time correction applied.
Gold row count: 4773


In [0]:
display(
    spark.table(gold_table)
    .select(
        "order_id",
        "line_id",
        "ordered_quantity",
        "uncapped_delivered_units",
        "capped_delivered_units",
        "fulfilment_percentage",
        "open_units",
        "completion_time",
        "is_overdelivered"
    )
    .limit(20)
)

order_id,line_id,ordered_quantity,uncapped_delivered_units,capped_delivered_units,fulfilment_percentage,open_units,completion_time,is_overdelivered
ORD-000001,L01,40,40,40,100.0,0,2026-08-02T03:45:30.000Z,false
ORD-000001,L02,20,20,20,100.0,0,2026-08-02T03:45:30.000Z,false
ORD-000002,L01,50,50,50,100.0,0,2026-08-04T13:25:56.000Z,false
ORD-000002,L02,60,60,60,100.0,0,2026-08-04T13:25:56.000Z,false
ORD-000003,L01,10,7,7,70.0,3,null,false
ORD-000003,L02,100,33,33,33.0,67,null,false
ORD-000003,L03,40,10,10,25.0,30,null,false
ORD-000003,L04,100,80,80,80.0,20,null,false
ORD-000003,L05,10,1,1,10.0,9,null,false
ORD-000003,L06,200,175,175,87.5,25,null,false


In [0]:

# PHASE 5 — STEP 3
# Build order_service_performance

from pyspark.sql import functions as F

catalog_name = "dbwteam3"
schema_name = "order_fulfillment"

as_of_time = "2026-10-01T00:00:00"

orders = spark.table(
    f"{catalog_name}.{schema_name}.silver_orders"
)

lines = spark.table(
    f"{catalog_name}.{schema_name}.order_line_fulfilment"
)

# 1. Aggregate line metrics to one row per order
order_line_summary = (
    lines
    .groupBy("order_id")
    .agg(
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias("total_capped_delivered_units"),
        F.sum("uncapped_delivered_units").alias("total_uncapped_delivered_units"),
        F.sum("open_units").alias("total_open_units"),
        F.max("is_overdelivered").alias("has_overdelivery"),
        F.min(
            F.when(F.col("open_units") > 0, F.lit(0))
            .otherwise(F.lit(1))
        ).alias("all_lines_complete"),
        F.max("completion_time").alias("latest_line_completion_time")
    )
    .withColumn(
        "order_fulfilment_percentage",
        F.round(
            F.col("total_capped_delivered_units") /
            F.col("total_ordered_units") * 100,
            2
        )
    )
)

# 2. Attach promised time, status, and order creation time
order_metrics = (
    orders
    .join(order_line_summary, "order_id", "left")
    .withColumn(
        "is_cancelled",
        F.lower(F.trim(F.col("order_status"))) == "cancelled"
    )
    .withColumn(
        "is_promised_by_as_of",
        F.col("promised_delivery_time") <= F.to_timestamp(
            F.lit(as_of_time)
        )
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled")) &
        F.col("is_promised_by_as_of") &
        (F.col("total_open_units") > 0)
    )
    .withColumn(
        "otif_pass",
        (~F.col("is_cancelled")) &
        F.col("is_promised_by_as_of") &
        (F.col("all_lines_complete") == 1) &
        (F.col("latest_line_completion_time") <= F.col("promised_delivery_time"))
    )
)

# 3. Calculate OTIF denominator eligibility
order_metrics = order_metrics.withColumn(
    "otif_denominator_eligible",
    (~F.col("is_cancelled")) &
    F.col("is_promised_by_as_of")
)

# 4. Persist the Gold order-level table
gold_table = (
    f"{catalog_name}.{schema_name}.order_service_performance"
)

(
    order_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print("Created:", gold_table)
print("Row count:", spark.table(gold_table).count())

display(
    spark.table(gold_table).select(
        "order_id",
        "order_status",
        "total_ordered_units",
        "total_capped_delivered_units",
        "total_open_units",
        "order_fulfilment_percentage",
        "is_overdue",
        "otif_denominator_eligible",
        "otif_pass"
    ).limit(20)
)


Created: dbwteam3.order_fulfillment.order_service_performance
Row count: 2040


order_id,order_status,total_ordered_units,total_capped_delivered_units,total_open_units,order_fulfilment_percentage,is_overdue,otif_denominator_eligible,otif_pass
ORD-000001,COMPLETED,60,60,0,100.0,false,true,true
ORD-000002,COMPLETED,110,110,0,100.0,false,true,true
ORD-000003,OPEN,460,306,154,66.52,true,true,false
ORD-000004,OPEN,520,216,304,41.54,true,true,false
ORD-000005,COMPLETED,20,20,0,100.0,false,true,true
ORD-000006,OPEN,56,27,29,48.21,true,true,false
ORD-000007,COMPLETED,230,230,0,100.0,false,true,true
ORD-000008,COMPLETED,30,30,0,100.0,false,true,true
ORD-000009,OPEN,60,60,0,100.0,false,true,false
ORD-000010,COMPLETED,60,60,0,100.0,false,true,true


In [0]:

from pyspark.sql import functions as F

catalog = "dbwteam3"
schema = "order_fulfillment"
as_of_time = "2026-10-01T00:00:00"

orders = spark.table(f"{catalog}.{schema}.silver_orders")
lines = spark.table(f"{catalog}.{schema}.order_line_fulfilment")

# Evaluate each line independently against the promised time.
line_otif = (
    lines
    .join(
        orders.select(
            "order_id",
            "promised_delivery_time",
            "order_status"
        ),
        "order_id",
        "inner"
    )
    .groupBy(
        "order_id",
        "promised_delivery_time",
        "order_status"
    )
    .agg(
        F.min(
            F.when(F.col("open_units") > 0, 0)
             .otherwise(1)
        ).alias("all_lines_complete"),
        F.min(
            F.when(
                (F.col("open_units") == 0) &
                (F.col("completion_time") <=
                 F.col("promised_delivery_time")),
                1
            ).otherwise(0)
        ).alias("all_lines_on_time"),
        F.sum("open_units").alias("total_open_units"),
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias(
            "total_capped_delivered_units"
        ),
        F.sum("uncapped_delivered_units").alias(
            "total_uncapped_delivered_units"
        ),
        F.max("is_overdelivered").alias("has_overdelivery")
    )
)

corrected = (
    line_otif
    .withColumn(
        "order_fulfilment_percentage",
        F.round(
            F.col("total_capped_delivered_units") /
            F.col("total_ordered_units") * 100,
            2
        )
    )
    .withColumn(
        "is_cancelled",
        F.lower(F.trim(F.col("order_status"))) == "cancelled"
    )
    .withColumn(
        "is_promised_by_as_of",
        F.col("promised_delivery_time") <=
        F.to_timestamp(F.lit(as_of_time))
    )
    .withColumn(
        "otif_denominator_eligible",
        (~F.col("is_cancelled")) & F.col("is_promised_by_as_of")
    )
    .withColumn(
        "otif_pass",
        F.col("otif_denominator_eligible") &
        (F.col("all_lines_complete") == 1) &
        (F.col("all_lines_on_time") == 1)
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled")) &
        F.col("is_promised_by_as_of") &
        (F.col("total_open_units") > 0)
    )
)

target = f"{catalog}.{schema}.order_service_performance"

(
    corrected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

print("Corrected order service performance table.")
print("Rows:", spark.table(target).count())

display(
    spark.table(target).select(
        "order_id",
        "order_status",
        "order_fulfilment_percentage",
        "total_open_units",
        "all_lines_complete",
        "all_lines_on_time",
        "otif_denominator_eligible",
        "otif_pass",
        "is_overdue"
    ).limit(20)
)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8240400201613529>, line 97
     53 corrected = (
     54     line_otif
     55     .withColumn(
   (...)     87     )
     88 )
     90 target = f"{catalog}.{schema}.order_service_performance"
     92 (
     93     corrected.write
     94     .format("delta")
     95     .mode("overwrite")
     96     .option("overwriteSchema", "true")
---> 97     .saveAsTable(target)
     98 )
    100 print("Corrected order service performance table.")
    101 print("Rows:", spark.table(target).count())

File <command-8240400201613529>, line 54
     11 # Evaluate each line independently against the promised time.
     12 line_otif = (
     13     lines
     14     .join(
   (...)     50     )
     51 )
     53 corrected = (
---> 54     line_otif
     55     .withColumn(
     56         "order_fulfilment_percentage",
     57         F.roun

In [0]:

from pyspark.sql import functions as F

catalog = "dbwteam3"
schema = "order_fulfillment"
as_of_time = "2026-10-01T00:00:00"

orders = spark.table(f"{catalog}.{schema}.silver_orders").select(
    "order_id",
    F.col("promised_delivery_time").alias("order_promised_time"),
    F.col("order_status").alias("order_status")
)

lines = spark.table(f"{catalog}.{schema}.order_line_fulfilment")

line_otif = (
    lines
    .join(orders, on="order_id", how="inner")
    .groupBy("order_id", "order_promised_time", "order_status")
    .agg(
        F.min(
            F.when(F.col("open_units") > 0, 0).otherwise(1)
        ).alias("all_lines_complete"),

        F.min(
            F.when(
                (F.col("open_units") == 0) &
                (F.col("completion_time") <= F.col("order_promised_time")),
                1
            ).otherwise(0)
        ).alias("all_lines_on_time"),

        F.sum("open_units").alias("total_open_units"),
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias("total_capped_delivered_units"),
        F.sum("uncapped_delivered_units").alias("total_uncapped_delivered_units"),
        F.max(
            F.col("is_overdelivered").cast("int")
        ).alias("has_overdelivery")
    )
)

corrected = (
    line_otif
    .withColumn(
        "order_fulfilment_percentage",
        F.round(
            F.col("total_capped_delivered_units") /
            F.col("total_ordered_units") * 100, 2
        )
    )
    .withColumn(
        "is_cancelled",
        F.lower(F.trim(F.col("order_status"))) == "cancelled"
    )
    .withColumn(
        "is_promised_by_as_of",
        F.col("order_promised_time") <= F.to_timestamp(F.lit(as_of_time))
    )
    .withColumn(
        "otif_denominator_eligible",
        (~F.col("is_cancelled")) & F.col("is_promised_by_as_of")
    )
    .withColumn(
        "otif_pass",
        F.col("otif_denominator_eligible") &
        (F.col("all_lines_complete") == 1) &
        (F.col("all_lines_on_time") == 1)
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled")) &
        F.col("is_promised_by_as_of") &
        (F.col("total_open_units") > 0)
    )
    .withColumnRenamed("order_promised_time", "promised_delivery_time")
)

target = f"{catalog}.{schema}.order_service_performance"

(
    corrected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

print("Corrected order service performance table.")
print("Rows:", spark.table(target).count())

display(
    spark.table(target).select(
        "order_id",
        "order_status",
        "order_fulfilment_percentage",
        "total_open_units",
        "all_lines_complete",
        "all_lines_on_time",
        "otif_denominator_eligible",
        "otif_pass",
        "is_overdue"
    ).limit(20)
)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8240400201613530>, line 86
     43 corrected = (
     44     line_otif
     45     .withColumn(
   (...)     76     .withColumnRenamed("order_promised_time", "promised_delivery_time")
     77 )
     79 target = f"{catalog}.{schema}.order_service_performance"
     81 (
     82     corrected.write
     83     .format("delta")
     84     .mode("overwrite")
     85     .option("overwriteSchema", "true")
---> 86     .saveAsTable(target)
     87 )
     89 print("Corrected order service performance table.")
     90 print("Rows:", spark.table(target).count())

File <command-8240400201613530>, line 44
     14 lines = spark.table(f"{catalog}.{schema}.order_line_fulfilment")
     16 line_otif = (
     17     lines
     18     .join(orders, on="order_id", how="inner")
   (...)     40     )
     41 )
     43 corrected = (
---> 44     

In [0]:

from pyspark.sql import functions as F

catalog = "dbwteam3"
schema = "order_fulfillment"
as_of_time = "2026-10-01T00:00:00"

# Select only required columns and rename order-level fields
orders = spark.table(f"{catalog}.{schema}.silver_orders").select(
    "order_id",
    F.col("promised_delivery_time").alias("order_promised_time"),
    F.col("order_status").alias("source_order_status")
)

lines = spark.table(f"{catalog}.{schema}.order_line_fulfilment").select(
    "order_id",
    "line_id",
    "ordered_quantity",
    "capped_delivered_units",
    "uncapped_delivered_units",
    "open_units",
    "completion_time",
    "is_overdelivered"
)

line_otif = (
    lines
    .join(orders, on="order_id", how="inner")
    .groupBy("order_id", "order_promised_time", "source_order_status")
    .agg(
        F.min(
            F.when(F.col("open_units") > 0, 0).otherwise(1)
        ).alias("all_lines_complete"),

        F.min(
            F.when(
                (F.col("open_units") == 0) &
                (F.col("completion_time") <= F.col("order_promised_time")),
                1
            ).otherwise(0)
        ).alias("all_lines_on_time"),

        F.sum("open_units").alias("total_open_units"),
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias("total_capped_delivered_units"),
        F.sum("uncapped_delivered_units").alias("total_uncapped_delivered_units"),
        F.max(F.col("is_overdelivered").cast("int")).alias("has_overdelivery")
    )
)

corrected = (
    line_otif
    .withColumn(
        "order_fulfilment_percentage",
        F.round(
            F.col("total_capped_delivered_units") /
            F.col("total_ordered_units") * 100, 2
        )
    )
    .withColumn(
        "is_cancelled",
        F.lower(F.trim(F.col("source_order_status"))) == "cancelled"
    )
    .withColumn(
        "is_promised_by_as_of",
        F.col("order_promised_time") <= F.to_timestamp(F.lit(as_of_time))
    )
    .withColumn(
        "otif_denominator_eligible",
        (~F.col("is_cancelled")) & F.col("is_promised_by_as_of")
    )
    .withColumn(
        "otif_pass",
        F.col("otif_denominator_eligible") &
        (F.col("all_lines_complete") == 1) &
        (F.col("all_lines_on_time") == 1)
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled")) &
        F.col("is_promised_by_as_of") &
        (F.col("total_open_units") > 0)
    )
    .withColumnRenamed("source_order_status", "order_status")
    .withColumnRenamed("order_promised_time", "promised_delivery_time")
)

target = f"{catalog}.{schema}.order_service_performance"

(
    corrected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

result = spark.table(target)
print("Corrected order service performance table.")
print("Rows:", result.count())

display(
    result.select(
        "order_id",
        "order_status",
        "order_fulfilment_percentage",
        "total_open_units",
        "all_lines_complete",
        "all_lines_on_time",
        "otif_denominator_eligible",
        "otif_pass",
        "is_overdue"
    ).limit(20)
)


Corrected order service performance table.
Rows: 2000


order_id,order_status,order_fulfilment_percentage,total_open_units,all_lines_complete,all_lines_on_time,otif_denominator_eligible,otif_pass,is_overdue
ORD-000001,COMPLETED,100.0,0,1,1,true,true,false
ORD-000006,OPEN,48.21,29,0,0,true,false,true
ORD-000013,CANCELLED,0.0,332,0,0,false,false,false
ORD-000015,OPEN,100.0,0,1,0,true,false,false
ORD-000027,CANCELLED,0.0,50,0,0,false,false,false
ORD-000088,OPEN,62.31,49,0,0,true,false,true
ORD-000092,OPEN,26.07,207,0,0,true,false,true
ORD-000099,COMPLETED,100.0,0,1,1,true,true,false
ORD-000100,OPEN,17.79,171,0,0,true,false,true
ORD-000107,COMPLETED,100.0,0,1,1,true,true,false


In [0]:

from pyspark.sql import functions as F

orders = spark.table(
    "dbwteam3.order_fulfillment.silver_orders"
).select("order_id")

gold = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
).select("order_id")

print("Silver orders:", orders.count())
print("Gold service-performance orders:", gold.count())

missing_orders = orders.join(
    gold,
    on="order_id",
    how="left_anti"
)

print("Orders missing from Gold:", missing_orders.count())
display(missing_orders)


Silver orders: 2040
Gold service-performance orders: 2000
Orders missing from Gold: 40


order_id
ORD-900021
ORD-900022
ORD-900023
ORD-900024
ORD-900025
ORD-900026
ORD-900027
ORD-900028
ORD-900029
ORD-900030


In [0]:

from pyspark.sql import functions as F

catalog = "dbwteam3.order_fulfillment"

missing = (
    spark.table(f"{catalog}.silver_orders")
    .join(
        spark.table(f"{catalog}.order_service_performance").select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

line_counts = (
    spark.table(f"{catalog}.silver_order_lines")
    .groupBy("order_id")
    .agg(F.count("*").alias("line_count"))
)

investigation = (
    missing
    .join(line_counts, on="order_id", how="left")
    .withColumn("line_count", F.coalesce(F.col("line_count"), F.lit(0)))
    .select("order_id", "order_status", "line_count", "promised_delivery_time")
    .orderBy("order_id")
)

display(investigation)
display(investigation.groupBy("order_status", "line_count").count())


order_id,order_status,line_count,promised_delivery_time
ORD-900021,OPEN,0,2026-09-16T07:08:56.000Z
ORD-900022,OPEN,0,2026-08-05T12:21:31.000Z
ORD-900023,OPEN,0,2026-08-09T08:00:12.000Z
ORD-900024,OPEN,0,2026-09-06T20:31:42.000Z
ORD-900025,OPEN,0,2026-08-05T13:31:12.000Z
ORD-900026,OPEN,0,2026-08-10T02:27:19.000Z
ORD-900027,OPEN,0,2026-08-18T01:35:19.000Z
ORD-900028,OPEN,0,2026-08-16T00:31:23.000Z
ORD-900029,OPEN,0,2026-08-06T08:51:19.000Z
ORD-900030,OPEN,0,2026-08-09T04:36:01.000Z


order_status,line_count,count
OPEN,0,40


In [0]:

from pyspark.sql import functions as F

catalog = "dbwteam3"
schema = "order_fulfillment"
as_of_time = "2026-10-01T00:00:00"

orders = spark.table(f"{catalog}.{schema}.silver_orders").select(
    "order_id",
    "promised_delivery_time",
    "order_status"
)

lines = spark.table(f"{catalog}.{schema}.order_line_fulfilment")

line_metrics = (
    lines
    .groupBy("order_id")
    .agg(
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias("total_capped_delivered_units"),
        F.sum("uncapped_delivered_units").alias("total_uncapped_delivered_units"),
        F.sum("open_units").alias("total_open_units"),
        F.min(
            F.when(F.col("open_units") > 0, 0).otherwise(1)
        ).alias("all_lines_complete"),
        F.min(
            F.when(
                (F.col("open_units") == 0) &
                (F.col("completion_time") <= F.col("promised_delivery_time")),
                1
            ).otherwise(0)
        ).alias("all_lines_on_time"),
        F.max(F.col("is_overdelivered").cast("int")).alias("has_overdelivery")
    )
)

# Use the order-level promised time for the OTIF comparison.
# Rename it before joining to avoid ambiguous column names.
order_data = orders.withColumnRenamed(
    "promised_delivery_time", "order_promised_time"
)

# Calculate line on-time status using the order's promised time.
line_on_time = (
    lines.select(
        "order_id", "open_units", "completion_time"
    )
    .join(
        order_data.select("order_id", "order_promised_time"),
        on="order_id",
        how="inner"
    )
    .groupBy("order_id")
    .agg(
        F.min(
            F.when(
                (F.col("open_units") == 0) &
                (F.col("completion_time") <= F.col("order_promised_time")),
                1
            ).otherwise(0)
        ).alias("all_lines_on_time")
    )
)

# Rebuild line metrics without a duplicate all_lines_on_time field.
line_metrics = (
    lines.groupBy("order_id")
    .agg(
        F.sum("ordered_quantity").alias("total_ordered_units"),
        F.sum("capped_delivered_units").alias("total_capped_delivered_units"),
        F.sum("uncapped_delivered_units").alias("total_uncapped_delivered_units"),
        F.sum("open_units").alias("total_open_units"),
        F.min(
            F.when(F.col("open_units") > 0, 0).otherwise(1)
        ).alias("all_lines_complete"),
        F.max(F.col("is_overdelivered").cast("int")).alias("has_overdelivery")
    )
    .join(line_on_time, on="order_id", how="inner")
)

corrected = (
    order_data
    .join(line_metrics, on="order_id", how="left")
    .withColumn(
        "total_ordered_units",
        F.coalesce(F.col("total_ordered_units"), F.lit(0))
    )
    .withColumn(
        "total_capped_delivered_units",
        F.coalesce(F.col("total_capped_delivered_units"), F.lit(0))
    )
    .withColumn(
        "total_uncapped_delivered_units",
        F.coalesce(F.col("total_uncapped_delivered_units"), F.lit(0))
    )
    .withColumn(
        "total_open_units",
        F.coalesce(F.col("total_open_units"), F.lit(0))
    )
    .withColumn(
        "has_overdelivery",
        F.coalesce(F.col("has_overdelivery"), F.lit(0))
    )
    .withColumn(
        "all_lines_complete",
        F.coalesce(F.col("all_lines_complete"), F.lit(0))
    )
    .withColumn(
        "all_lines_on_time",
        F.coalesce(F.col("all_lines_on_time"), F.lit(0))
    )
    .withColumn(
        "order_fulfilment_percentage",
        F.when(
            F.col("total_ordered_units") > 0,
            F.round(
                F.col("total_capped_delivered_units") /
                F.col("total_ordered_units") * 100, 2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "is_cancelled",
        F.lower(F.trim(F.col("order_status"))) == "cancelled"
    )
    .withColumn(
        "is_promised_by_as_of",
        F.col("order_promised_time") <= F.to_timestamp(F.lit(as_of_time))
    )
    .withColumn(
        "otif_denominator_eligible",
        (~F.col("is_cancelled")) & F.col("is_promised_by_as_of")
    )
    .withColumn(
        "otif_pass",
        F.col("otif_denominator_eligible") &
        (F.col("all_lines_complete") == 1) &
        (F.col("all_lines_on_time") == 1) &
        (F.col("total_ordered_units") > 0)
    )
    .withColumn(
        "is_overdue",
        (~F.col("is_cancelled")) &
        F.col("is_promised_by_as_of") &
        (F.col("total_open_units") > 0)
    )
    .withColumnRenamed("order_promised_time", "promised_delivery_time")
)

target = f"{catalog}.{schema}.order_service_performance"

(
    corrected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

result = spark.table(target)

print("Gold order count:", result.count())
print("Missing source orders:", 
      spark.table(f"{catalog}.{schema}.silver_orders")
      .select("order_id")
      .join(result.select("order_id"), "order_id", "left_anti")
      .count())

display(
    result.filter(F.col("order_id").between("ORD-900021", "ORD-900060"))
    .select(
        "order_id", "order_status", "total_ordered_units",
        "order_fulfilment_percentage", "total_open_units",
        "otif_denominator_eligible", "otif_pass", "is_overdue"
    )
    .orderBy("order_id")
)


Gold order count: 2040
Missing source orders: 0


order_id,order_status,total_ordered_units,order_fulfilment_percentage,total_open_units,otif_denominator_eligible,otif_pass,is_overdue
ORD-900021,OPEN,0,null,0,true,false,false
ORD-900022,OPEN,0,null,0,true,false,false
ORD-900023,OPEN,0,null,0,true,false,false
ORD-900024,OPEN,0,null,0,true,false,false
ORD-900025,OPEN,0,null,0,true,false,false
ORD-900026,OPEN,0,null,0,true,false,false
ORD-900027,OPEN,0,null,0,true,false,false
ORD-900028,OPEN,0,null,0,true,false,false
ORD-900029,OPEN,0,null,0,true,false,false
ORD-900030,OPEN,0,null,0,true,false,false


In [0]:

from pyspark.sql import functions as F

target = "dbwteam3.order_fulfillment.order_service_performance"

df = spark.table(target)

fixed = df.withColumn(
    "is_overdue",
    (~F.col("is_cancelled")) &
    F.col("is_promised_by_as_of") &
    (
        (F.col("total_open_units") > 0) |
        (F.col("total_ordered_units") == 0)
    )
)

(
    fixed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

result = spark.table(target)

print("Gold order count:", result.count())
print("Overdue orders:", result.filter(F.col("is_overdue")).count())

display(
    result.filter(
        F.col("order_id").between("ORD-900021", "ORD-900060")
    ).select(
        "order_id",
        "order_status",
        "total_ordered_units",
        "is_promised_by_as_of",
        "is_overdue",
        "otif_pass"
    ).orderBy("order_id")
)


Gold order count: 2040
Overdue orders: 540


order_id,order_status,total_ordered_units,is_promised_by_as_of,is_overdue,otif_pass
ORD-900021,OPEN,0,true,true,false
ORD-900022,OPEN,0,true,true,false
ORD-900023,OPEN,0,true,true,false
ORD-900024,OPEN,0,true,true,false
ORD-900025,OPEN,0,true,true,false
ORD-900026,OPEN,0,true,true,false
ORD-900027,OPEN,0,true,true,false
ORD-900028,OPEN,0,true,true,false
ORD-900029,OPEN,0,true,true,false
ORD-900030,OPEN,0,true,true,false


In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

df.select(
    F.count("*").alias("total_orders"),

    F.sum(
        F.col("otif_denominator_eligible").cast("int")
    ).alias("otif_eligible_orders"),

    F.sum(
        F.col("otif_pass").cast("int")
    ).alias("otif_pass_orders"),

    F.round(
        100.0 * F.sum(F.col("otif_pass").cast("int")) /
        F.nullif(
            F.sum(F.col("otif_denominator_eligible").cast("int")), 0
        ),
        2
    ).alias("otif_percentage"),

    F.sum(
        F.col("is_overdue").cast("int")
    ).alias("overdue_orders"),

    F.sum(
        F.when(F.col("total_ordered_units") == 0, 1).otherwise(0)
    ).alias("orders_without_lines")
).show(truncate=False)


---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-8240400201613536>, line 21
      2 from pyspark.sql import functions as F
      4 df = spark.table(
      5     "dbwteam3.order_fulfillment.order_service_performance"
      6 )
      8 df.select(
      9     F.count("*").alias("total_orders"),
     10 
     11     F.sum(
     12         F.col("otif_denominator_eligible").cast("int")
     13     ).alias("otif_eligible_orders"),
     14 
     15     F.sum(
     16         F.col("otif_pass").cast("int")
     17     ).alias("otif_pass_orders"),
     18 
     19     F.round(
     20         100.0 * F.sum(F.col("otif_pass").cast("int")) /
---> 21         F.nullif(
     22             F.sum(F.col("otif_denominator_eligible").cast("int")), 0
     23         ),
     24         2
     25     ).alias("otif_percentage"),
     26 
     27     F.sum(
     28         F.col("is_overdue").

In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

summary = df.agg(
    F.count("*").alias("total_orders"),

    F.sum(
        F.col("otif_denominator_eligible").cast("int")
    ).alias("otif_eligible_orders"),

    F.sum(
        F.col("otif_pass").cast("int")
    ).alias("otif_pass_orders"),

    F.sum(
        F.col("is_overdue").cast("int")
    ).alias("overdue_orders"),

    F.sum(
        F.when(F.col("total_ordered_units") == 0, 1).otherwise(0)
    ).alias("orders_without_lines")
)

summary = summary.withColumn(
    "otif_percentage",
    F.when(
        F.col("otif_eligible_orders") > 0,
        F.round(
            F.col("otif_pass_orders") * F.lit(100.0) /
            F.col("otif_eligible_orders"),
            2
        )
    ).otherwise(F.lit(None).cast("double"))
)

summary.select(
    "total_orders",
    "otif_eligible_orders",
    "otif_pass_orders",
    "otif_percentage",
    "overdue_orders",
    "orders_without_lines"
).show(truncate=False)


+------------+--------------------+----------------+---------------+--------------+--------------------+
|total_orders|otif_eligible_orders|otif_pass_orders|otif_percentage|overdue_orders|orders_without_lines|
+------------+--------------------+----------------+---------------+--------------+--------------------+
|2040        |1940                |1000            |51.55          |540           |40                  |
+------------+--------------------+----------------+---------------+--------------+--------------------+



In [0]:

from pyspark.sql import functions as F

catalog = "dbwteam3"
schema = "order_fulfillment"

orders = spark.table(
    f"{catalog}.{schema}.order_service_performance"
)

pharmacies = spark.table(
    f"{catalog}.{schema}.silver_pharmacies"
).select("pharmacy_id", "region")

# Attach pharmacy and regional information to each order.
order_regions = (
    orders
    .join(
        spark.table(f"{catalog}.{schema}.silver_orders")
        .select("order_id", "pharmacy_id"),
        on="order_id",
        how="left"
    )
    .join(pharmacies, on="pharmacy_id", how="left")
)

# Aggregate backlog metrics by region.
regional_backlog = (
    order_regions
    .groupBy("region")
    .agg(
        F.sum("total_open_units").alias("total_open_units"),

        F.sum(
            F.when(F.col("total_open_units") > 0, 1).otherwise(0)
        ).alias("orders_with_open_units"),

        F.sum(
            F.col("is_overdue").cast("int")
        ).alias("overdue_orders"),

        F.sum(
            F.when(F.col("total_ordered_units") == 0, 1).otherwise(0)
        ).alias("orders_without_lines")
    )
    .withColumn(
        "as_of_time",
        F.to_timestamp(F.lit("2026-10-01T00:00:00"))
    )
)

target = f"{catalog}.{schema}.regional_backlog"

(
    regional_backlog.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

print("Regional backlog table created.")
print("Rows:", spark.table(target).count())

display(
    spark.table(target).orderBy(F.desc("total_open_units"))
)


Regional backlog table created.
Rows: 5


region,total_open_units,orders_with_open_units,overdue_orders,orders_without_lines,as_of_time
West,12261,134,121,6,2026-10-01T00:00:00.000Z
North,11930,140,128,10,2026-10-01T00:00:00.000Z
Central,11170,103,87,6,2026-10-01T00:00:00.000Z
East,11136,108,105,11,2026-10-01T00:00:00.000Z
South,10581,115,99,7,2026-10-01T00:00:00.000Z


In [0]:

from pyspark.sql import functions as F

schema = "dbwteam3.order_fulfillment"

orders = spark.table(f"{schema}.order_service_performance")
regional = spark.table(f"{schema}.regional_backlog")

# Calculate expected totals directly from the order-level Gold table.
expected = orders.agg(
    F.sum("total_open_units").alias("open_units"),
    F.sum(F.when(F.col("total_open_units") > 0, 1).otherwise(0))
        .alias("orders_with_open_units"),
    F.sum(F.col("is_overdue").cast("int")).alias("overdue_orders"),
    F.sum(F.when(F.col("total_ordered_units") == 0, 1).otherwise(0))
        .alias("orders_without_lines")
)

# Calculate totals from the regional table.
actual = regional.agg(
    F.sum("total_open_units").alias("open_units"),
    F.sum("orders_with_open_units").alias("orders_with_open_units"),
    F.sum("overdue_orders").alias("overdue_orders"),
    F.sum("orders_without_lines").alias("orders_without_lines")
)

print("Expected totals from order-level Gold:")
expected.show(truncate=False)

print("Actual totals from regional backlog:")
actual.show(truncate=False)

print("Regional rows:", regional.count())


Expected totals from order-level Gold:
+----------+----------------------+--------------+--------------------+
|open_units|orders_with_open_units|overdue_orders|orders_without_lines|
+----------+----------------------+--------------+--------------------+
|57078     |600                   |540           |40                  |
+----------+----------------------+--------------+--------------------+

Actual totals from regional backlog:
+----------+----------------------+--------------+--------------------+
|open_units|orders_with_open_units|overdue_orders|orders_without_lines|
+----------+----------------------+--------------+--------------------+
|57078     |600                   |540           |40                  |
+----------+----------------------+--------------+--------------------+

Regional rows: 5


In [0]:

from pyspark.sql import functions as F

lines = spark.table(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)

# Validate fulfilment calculations and identify suspicious records.
checks = lines.agg(
    F.count("*").alias("total_order_lines"),

    F.sum(
        F.when(
            F.col("ordered_quantity") <= 0, 1
        ).otherwise(0)
    ).alias("nonpositive_ordered_quantity"),

    F.sum(
        F.when(
            F.col("capped_delivered_units") >
            F.col("ordered_quantity"), 1
        ).otherwise(0)
    ).alias("capped_delivery_exceeds_ordered"),

    F.sum(
        F.when(
            F.col("open_units") != F.greatest(
                F.lit(0),
                F.col("ordered_quantity") -
                F.col("capped_delivered_units")
            ), 1
        ).otherwise(0)
    ).alias("incorrect_open_unit_calculations"),

    F.sum(
        F.when(
            (F.col("open_units") == 0) &
            (F.col("fulfilment_percentage") != 100), 1
        ).otherwise(0)
    ).alias("complete_lines_not_at_100_percent")
)

checks.show(truncate=False)


+-----------------+----------------------------+-------------------------------+--------------------------------+---------------------------------+
|total_order_lines|nonpositive_ordered_quantity|capped_delivery_exceeds_ordered|incorrect_open_unit_calculations|complete_lines_not_at_100_percent|
+-----------------+----------------------------+-------------------------------+--------------------------------+---------------------------------+
|4773             |0                           |0                              |0                               |0                                |
+-----------------+----------------------------+-------------------------------+--------------------------------+---------------------------------+



In [0]:

from pyspark.sql import functions as F

schema = "dbwteam3.order_fulfillment"

orders = spark.table(f"{schema}.order_service_performance")
lines = spark.table(f"{schema}.order_line_fulfilment")
silver_orders = spark.table(f"{schema}.silver_orders")

# Recalculate the expected line-level completion and on-time status.
line_checks = (
    lines
    .join(
        silver_orders.select(
            "order_id",
            F.col("promised_delivery_time").alias("order_promised_time")
        ),
        on="order_id",
        how="inner"
    )
    .groupBy("order_id")
    .agg(
        F.min(
            F.when(F.col("open_units") > 0, 0).otherwise(1)
        ).alias("expected_all_lines_complete"),

        F.min(
            F.when(
                (F.col("open_units") == 0) &
                (F.col("completion_time") <= F.col("order_promised_time")),
                1
            ).otherwise(0)
        ).alias("expected_all_lines_on_time")
    )
)

checked = (
    orders
    .join(line_checks, on="order_id", how="left")
    .withColumn(
        "expected_otif_pass",
        F.col("otif_denominator_eligible") &
        (F.coalesce(F.col("expected_all_lines_complete"), F.lit(0)) == 1) &
        (F.coalesce(F.col("expected_all_lines_on_time"), F.lit(0)) == 1)
    )
)

checked.select(
    F.sum(
        F.when(
            F.col("otif_pass") != F.col("expected_otif_pass"), 1
        ).otherwise(0)
    ).alias("otif_pass_mismatches"),

    F.sum(
        F.when(
            F.col("otif_pass") &
            (F.coalesce(F.col("expected_all_lines_complete"), F.lit(0)) == 0),
            1
        ).otherwise(0)
    ).alias("passing_orders_with_incomplete_lines"),

    F.sum(
        F.when(
            F.col("otif_pass") &
            (F.coalesce(F.col("expected_all_lines_on_time"), F.lit(0)) == 0),
            1
        ).otherwise(0)
    ).alias("passing_orders_with_late_lines")
).show(truncate=False)


+--------------------+------------------------------------+------------------------------+
|otif_pass_mismatches|passing_orders_with_incomplete_lines|passing_orders_with_late_lines|
+--------------------+------------------------------------+------------------------------+
|0                   |0                                   |0                             |
+--------------------+------------------------------------+------------------------------+



In [0]:

from pyspark.sql import functions as F

events = spark.table(
    "dbwteam3.order_fulfillment.silver_delivery_events"
)

events.agg(
    F.count("*").alias("total_events"),
    F.countDistinct("event_id").alias("unique_event_ids"),
    F.sum(
        F.when(F.col("event_id").isNull(), 1).otherwise(0)
    ).alias("null_event_ids")
).show(truncate=False)

duplicates = (
    events.groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate event IDs:", duplicates.count())
display(duplicates)


+------------+----------------+--------------+
|total_events|unique_event_ids|null_event_ids|
+------------+----------------+--------------+
|6018        |6018            |0             |
+------------+----------------+--------------+

Duplicate event IDs: 0


event_id,count


In [0]:

from pyspark.sql import functions as F

# Isolated acceptance-test scenarios; these do not modify Gold tables.
test_cases = [
    # Case, ordered, delivered, promise met, cancelled, expected fulfilment, expected OTIF
    ("partial_delivery",       100, 60,  True,  False, 60.0,  False),
    ("late_final_delivery",    100, 100, False, False, 100.0, False),
    ("one_line_late",          200, 200, False, False, 100.0, False),
    ("cancelled_order",        100, 100, True,  True,  100.0, False),
]

test_df = spark.createDataFrame(
    test_cases,
    [
        "test_case", "ordered_units", "delivered_units",
        "all_lines_on_time", "is_cancelled",
        "expected_fulfilment_pct", "expected_otif_pass"
    ]
)

results = (
    test_df
    .withColumn(
        "actual_fulfilment_pct",
        F.when(
            F.col("ordered_units") > 0,
            F.least(
                F.col("delivered_units"),
                F.col("ordered_units")
            ) / F.col("ordered_units") * 100
        )
    )
    .withColumn(
        "actual_otif_pass",
        (~F.col("is_cancelled"))
        & (F.col("delivered_units") >= F.col("ordered_units"))
        & F.col("all_lines_on_time")
    )
    .withColumn(
        "test_passed",
        (F.abs(
            F.col("actual_fulfilment_pct")
            - F.col("expected_fulfilment_pct")
        ) < 0.001)
        & (F.col("actual_otif_pass") == F.col("expected_otif_pass"))
    )
)

results.select(
    "test_case",
    "actual_fulfilment_pct",
    "actual_otif_pass",
    "test_passed"
).show(truncate=False)

failed_tests = results.filter(~F.col("test_passed")).count()
assert failed_tests == 0, f"{failed_tests} acceptance test(s) failed"

print("All four isolated acceptance tests passed.")


+-------------------+---------------------+----------------+-----------+
|test_case          |actual_fulfilment_pct|actual_otif_pass|test_passed|
+-------------------+---------------------+----------------+-----------+
|partial_delivery   |60.0                 |false           |true       |
|late_final_delivery|100.0                |false           |true       |
|one_line_late      |100.0                |false           |true       |
|cancelled_order    |100.0                |false           |true       |
+-------------------+---------------------+----------------+-----------+

All four isolated acceptance tests passed.


In [0]:

from pyspark.sql import functions as F

gold_orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

gold_orders.groupBy("is_cancelled").agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.col("otif_denominator_eligible").cast("int")
    ).alias("otif_eligible_orders"),
    F.sum(
        F.col("otif_pass").cast("int")
    ).alias("otif_pass_orders")
).orderBy("is_cancelled").show(truncate=False)

cancelled_eligible = gold_orders.filter(
    F.col("is_cancelled")
    & F.col("otif_denominator_eligible")
).count()

assert cancelled_eligible == 0, (
    f"Found {cancelled_eligible} cancelled orders in the OTIF denominator"
)

print("PASS: Cancelled orders are excluded from the OTIF denominator.")


+------------+------------+--------------------+----------------+
|is_cancelled|total_orders|otif_eligible_orders|otif_pass_orders|
+------------+------------+--------------------+----------------+
|false       |1940        |1940                |1000            |
|true        |100         |0                   |0               |
+------------+------------+--------------------+----------------+

PASS: Cancelled orders are excluded from the OTIF denominator.


In [0]:

from pyspark.sql import functions as F

gold_lines = spark.table(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)

gold_orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

# Find orders marked as OTIF passes that have at least one
# incomplete or late line.
bad_lines = gold_lines.filter(
    (F.col("open_units") > 0)
    | (
        F.col("completion_time").isNotNull()
        & (F.col("completion_time") > F.col("promised_delivery_time"))
    )
)

# Join to the order table to identify any incorrect OTIF passes.
violations = (
    bad_lines
    .select("order_id")
    .distinct()
    .join(
        gold_orders.filter(F.col("otif_pass") == True)
        .select("order_id"),
        on="order_id",
        how="inner"
    )
)

violation_count = violations.count()

print(f"OTIF-pass orders with an incomplete or late line: {violation_count}")
assert violation_count == 0, (
    f"Found {violation_count} OTIF-pass orders with an incomplete or late line"
)

print("PASS: No OTIF-pass order has an incomplete or late line.")


OTIF-pass orders with an incomplete or late line: 0
PASS: No OTIF-pass order has an incomplete or late line.


In [0]:

from pyspark.sql import functions as F

gold_lines = spark.table(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)

gold_orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

late_completed_lines = (
    gold_lines.alias("l")
    .join(
        gold_orders.select(
            "order_id",
            "promised_delivery_time",
            "otif_pass"
        ).alias("o"),
        on="order_id",
        how="inner"
    )
    .filter(
        (F.col("l.open_units") == 0)
        & F.col("l.completion_time").isNotNull()
        & (
            F.col("l.completion_time")
            > F.col("o.promised_delivery_time")
        )
    )
)

print(
    "Fully delivered lines completed after promise:",
    late_completed_lines.count()
)

late_completed_lines.select(
    "order_id",
    "line_id",
    "capped_delivered_quantity",
    "open_units",
    "completion_time",
    "promised_delivery_time",
    "otif_pass"
).show(10, truncate=False)


Fully delivered lines completed after promise: 636


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7924714871508148>, line 46
     12 late_completed_lines = (
     13     gold_lines.alias("l")
     14     .join(
   (...)     30     )
     31 )
     33 print(
     34     "Fully delivered lines completed after promise:",
     35     late_completed_lines.count()
     36 )
     38 late_completed_lines.select(
     39     "order_id",
     40     "line_id",
     41     "capped_delivered_quantity",
     42     "open_units",
     43     "completion_time",
     44     "promised_delivery_time",
     45     "otif_pass"
---> 46 ).show(10, truncate=False)

File <command-7924714871508148>, line 13
      4 gold_lines = spark.table(
      5     "dbwteam3.order_fulfillment.order_line_fulfilment"
      6 )
      8 gold_orders = spark.table(
      9     "dbwteam3.order_fulfillment.order_service_performance"
     10 )
     12 late_complete

In [0]:

from pyspark.sql import functions as F

gold_lines = spark.table(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)

gold_orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

late_completed_lines = (
    gold_lines.alias("l")
    .join(
        gold_orders.select(
            "order_id",
            F.col("promised_delivery_time").alias("order_promise_time"),
            "otif_pass"
        ).alias("o"),
        on="order_id",
        how="inner"
    )
    .filter(
        (F.col("l.open_units") == 0)
        & F.col("l.completion_time").isNotNull()
        & (
            F.col("l.completion_time")
            > F.col("order_promise_time")
        )
    )
)

print(
    "Fully delivered lines completed after promise:",
    late_completed_lines.count()
)

late_completed_lines.select(
    "order_id",
    "line_id",
    "capped_delivered_quantity",
    "open_units",
    "completion_time",
    "order_promise_time",
    "otif_pass"
).show(10, truncate=False)


Fully delivered lines completed after promise: 636


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7924714871508149>, line 46
     12 late_completed_lines = (
     13     gold_lines.alias("l")
     14     .join(
   (...)     30     )
     31 )
     33 print(
     34     "Fully delivered lines completed after promise:",
     35     late_completed_lines.count()
     36 )
     38 late_completed_lines.select(
     39     "order_id",
     40     "line_id",
     41     "capped_delivered_quantity",
     42     "open_units",
     43     "completion_time",
     44     "order_promise_time",
     45     "otif_pass"
---> 46 ).show(10, truncate=False)

File <command-7924714871508149>, line 13
      4 gold_lines = spark.table(
      5     "dbwteam3.order_fulfillment.order_line_fulfilment"
      6 )
      8 gold_orders = spark.table(
      9     "dbwteam3.order_fulfillment.order_service_performance"
     10 )
     12 late_completed_li

In [0]:

from pyspark.sql import functions as F

gold_lines = spark.table(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)

gold_orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

late_completed_lines = (
    gold_lines.alias("l")
    .join(
        gold_orders.select(
            "order_id",
            F.col("promised_delivery_time").alias("order_promise_time"),
            "otif_pass"
        ).alias("o"),
        on="order_id",
        how="inner"
    )
    .filter(
        (F.col("l.open_units") == 0)
        & F.col("l.completion_time").isNotNull()
        & (
            F.col("l.completion_time")
            > F.col("order_promise_time")
        )
    )
)

print(
    "Fully delivered lines completed after promise:",
    late_completed_lines.count()
)

late_completed_lines.select(
    "order_id",
    "line_id",
    "capped_delivered_units",
    "open_units",
    "completion_time",
    "order_promise_time",
    "otif_pass"
).show(10, truncate=False)


Fully delivered lines completed after promise: 636
+----------+-------+----------------------+----------+-------------------+-------------------+---------+
|order_id  |line_id|capped_delivered_units|open_units|completion_time    |order_promise_time |otif_pass|
+----------+-------+----------------------+----------+-------------------+-------------------+---------+
|ORD-000009|L01    |60                    |0         |2026-08-06 10:03:52|2026-08-06 03:50:16|false    |
|ORD-000015|L02    |30                    |0         |2026-08-07 13:37:46|2026-08-06 09:18:17|false    |
|ORD-000031|L01    |60                    |0         |2026-08-05 19:31:26|2026-08-04 19:49:11|false    |
|ORD-000032|L01    |100                   |0         |2026-08-04 15:05:10|2026-08-03 20:49:05|false    |
|ORD-000038|L01    |120                   |0         |2026-08-04 19:57:13|2026-08-03 22:26:08|false    |
|ORD-000044|L02    |60                    |0         |2026-08-06 10:29:44|2026-08-05 01:36:50|false    |
|ORD

In [0]:

late_line_otif_violations = late_completed_lines.filter(
    F.col("o.otif_pass") == True
).count()

print("Late completed lines belonging to OTIF-pass orders:",
      late_line_otif_violations)

assert late_line_otif_violations == 0, (
    "FAIL: At least one late-completed line belongs to an OTIF-pass order."
)

print("PASS: All fully delivered late lines belong to OTIF-failing orders.")


Late completed lines belonging to OTIF-pass orders: 0
PASS: All fully delivered late lines belong to OTIF-failing orders.


In [0]:

from pyspark.sql import functions as F

gold_table_names = [
    "order_line_fulfilment",
    "order_service_performance",
    "regional_backlog"
]

baseline = {}

for table_name in gold_table_names:
    df = spark.table(f"dbwteam3.order_fulfillment.{table_name}")

    metrics = df.agg(
        F.count("*").alias("row_count"),
        F.sum("total_open_units").alias("total_open_units")
        if "total_open_units" in df.columns
        else F.lit(None).alias("total_open_units"),
        F.sum("otif_pass").alias("otif_pass_count")
        if "otif_pass" in df.columns
        else F.lit(None).alias("otif_pass_count"),
        F.sum("overdue_orders").alias("overdue_orders")
        if "overdue_orders" in df.columns
        else F.lit(None).alias("overdue_orders")
    ).first()

    baseline[table_name] = metrics.asDict()

    print(table_name, baseline[table_name])

print("Baseline captured. Do not rerun the jobs yet.")


order_line_fulfilment {'row_count': 4773, 'total_open_units': None, 'otif_pass_count': None, 'overdue_orders': None}


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7924714871508152>, line 26
     12 for table_name in gold_table_names:
     13     df = spark.table(f"dbwteam3.order_fulfillment.{table_name}")
     15     metrics = df.agg(
     16         F.count("*").alias("row_count"),
     17         F.sum("total_open_units").alias("total_open_units")
     18         if "total_open_units" in df.columns
     19         else F.lit(None).alias("total_open_units"),
     20         F.sum("otif_pass").alias("otif_pass_count")
     21         if "otif_pass" in df.columns
     22         else F.lit(None).alias("otif_pass_count"),
     23         F.sum("overdue_orders").alias("overdue_orders")
     24         if "overdue_orders" in df.columns
     25         else F.lit(None).alias("overdue_orders")
---> 26     ).first()
     28     baseline[table_name] = metrics.asDict()
     30     print(tabl

In [0]:

from pyspark.sql import functions as F

gold_table_names = [
    "order_line_fulfilment",
    "order_service_performance",
    "regional_backlog"
]

baseline = {}

for table_name in gold_table_names:
    table_path = f"dbwteam3.order_fulfillment.{table_name}"
    df = spark.table(table_path)

    metrics = {"row_count": df.count()}

    if "total_open_units" in df.columns:
        metrics["total_open_units"] = (
            df.agg(F.sum("total_open_units")).first()[0]
        )

    if "otif_pass" in df.columns:
        metrics["otif_pass_count"] = (
            df.agg(
                F.sum(F.col("otif_pass").cast("int"))
            ).first()[0]
        )

    if "overdue_orders" in df.columns:
        metrics["overdue_orders"] = (
            df.agg(F.sum("overdue_orders")).first()[0]
        )

    baseline[table_name] = metrics
    print(f"{table_name}: {metrics}")

print("Baseline captured for all three Gold tables.")


order_line_fulfilment: {'row_count': 4773}
order_service_performance: {'row_count': 2040, 'total_open_units': 57078, 'otif_pass_count': 1000}
regional_backlog: {'row_count': 5, 'total_open_units': 57078, 'overdue_orders': 540}
Baseline captured for all three Gold tables.


In [0]:

from pyspark.sql import functions as F

current_metrics = {}

for table_name in [
    "order_line_fulfilment",
    "order_service_performance",
    "regional_backlog"
]:
    df = spark.table(f"dbwteam3.order_fulfillment.{table_name}")

    metrics = {"row_count": df.count()}

    if "total_open_units" in df.columns:
        metrics["total_open_units"] = df.agg(
            F.sum("total_open_units")
        ).first()[0]

    if "otif_pass" in df.columns:
        metrics["otif_pass_count"] = df.agg(
            F.sum(F.col("otif_pass").cast("int"))
        ).first()[0]

    if "overdue_orders" in df.columns:
        metrics["overdue_orders"] = df.agg(
            F.sum("overdue_orders")
        ).first()[0]

    current_metrics[table_name] = metrics

for table_name in current_metrics:
    expected = baseline[table_name]
    actual = current_metrics[table_name]

    print(
        f"{table_name}: "
        f"{'PASS' if expected == actual else 'MISMATCH'}"
    )
    print("  Baseline:", expected)
    print("  Current: ", actual)

assert all(
    baseline[name] == current_metrics[name]
    for name in current_metrics
), "Gold metrics changed since baseline capture."

print("PASS: Current Gold metrics match the saved baseline.")


order_line_fulfilment: PASS
  Baseline: {'row_count': 4773}
  Current:  {'row_count': 4773}
order_service_performance: PASS
  Baseline: {'row_count': 2040, 'total_open_units': 57078, 'otif_pass_count': 1000}
  Current:  {'row_count': 2040, 'total_open_units': 57078, 'otif_pass_count': 1000}
regional_backlog: PASS
  Baseline: {'row_count': 5, 'total_open_units': 57078, 'overdue_orders': 540}
  Current:  {'row_count': 5, 'total_open_units': 57078, 'overdue_orders': 540}
PASS: Current Gold metrics match the saved baseline.


In [0]:

# Display the write/save commands in your notebook's current source
# by reviewing the Gold build cells you used to create the tables.

print("Inspect the Gold build cells for these patterns:")
print("- .mode('overwrite')")
print("- .mode('append')")
print("- MERGE INTO")
print("- CREATE OR REPLACE TABLE")
print("- INSERT INTO")


Inspect the Gold build cells for these patterns:
- .mode('overwrite')
- .mode('append')
- MERGE INTO
- CREATE OR REPLACE TABLE
- INSERT INTO


In [0]:




corrected_gold.write.mode("overwrite").saveAsTable(
    "dbwteam3.order_fulfillment.order_line_fulfilment"
)
fixed.write.mode("overwrite").saveAsTable(
    "dbwteam3.order_fulfillment.order_service_performance"
)

regional_backlog.write.mode("overwrite").saveAsTable(
    "dbwteam3.order_fulfillment.regional_backlog"
)

In [0]:

from pyspark.sql import functions as F

expected = {
    "order_line_fulfilment": {
        "row_count": 4773
    },
    "order_service_performance": {
        "row_count": 2040,
        "total_open_units": 57078,
        "otif_pass_count": 1000
    },
    "regional_backlog": {
        "row_count": 5,
        "total_open_units": 57078,
        "overdue_orders": 540
    }
}

for table_name, expected_metrics in expected.items():
    df = spark.table(
        f"dbwteam3.order_fulfillment.{table_name}"
    )

    actual = {"row_count": df.count()}

    if "total_open_units" in df.columns:
        actual["total_open_units"] = df.agg(
            F.sum("total_open_units")
        ).first()[0]

    if "otif_pass" in df.columns:
        actual["otif_pass_count"] = df.agg(
            F.sum(F.col("otif_pass").cast("int"))
        ).first()[0]

    if "overdue_orders" in df.columns:
        actual["overdue_orders"] = df.agg(
            F.sum("overdue_orders")
        ).first()[0]

    print(f"{table_name}: {actual}")

    for metric, expected_value in expected_metrics.items():
        assert actual.get(metric) == expected_value, (
            f"{table_name}: {metric} expected {expected_value}, "
            f"got {actual.get(metric)}"
        )

print("PASS: All three Gold tables match expected metrics.")


order_line_fulfilment: {'row_count': 4773}
order_service_performance: {'row_count': 2040, 'total_open_units': 57078, 'otif_pass_count': 1000}
regional_backlog: {'row_count': 5, 'total_open_units': 57078, 'overdue_orders': 540}
PASS: All three Gold tables match expected metrics.


In [0]:

from pyspark.sql import functions as F

orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

regional = spark.table(
    "dbwteam3.order_fulfillment.regional_backlog"
)

order_overdue = orders.filter(
    F.col("is_overdue") == True
).count()

regional_overdue = regional.agg(
    F.sum("overdue_orders")
).first()[0]

print("Order-level overdue orders:", order_overdue)
print("Regional summed overdue orders:", regional_overdue)
print("Difference:", order_overdue - regional_overdue)

assert order_overdue == 540, (
    f"Expected 540 order-level overdue orders, got {order_overdue}"
)


Order-level overdue orders: 540
Regional summed overdue orders: 540
Difference: 0


In [0]:

from pyspark.sql import functions as F

orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

orders.groupBy(
    "is_cancelled",
    "is_promised_by_as_of",
    "is_overdue"
).agg(
    F.count("*").alias("order_count"),
    F.sum(
        (F.col("total_ordered_units") == 0).cast("int")
    ).alias("orders_without_lines"),
    F.sum(
        (F.col("total_open_units") > 0).cast("int")
    ).alias("orders_with_open_units")
).orderBy(
    "is_cancelled",
    "is_promised_by_as_of",
    "is_overdue"
).show(truncate=False)


+------------+--------------------+----------+-----------+--------------------+----------------------+
|is_cancelled|is_promised_by_as_of|is_overdue|order_count|orders_without_lines|orders_with_open_units|
+------------+--------------------+----------+-----------+--------------------+----------------------+
|false       |true                |false     |1400       |0                   |0                     |
|false       |true                |true      |540        |40                  |500                   |
|true        |true                |false     |100        |0                   |100                   |
+------------+--------------------+----------+-----------+--------------------+----------------------+



In [0]:

from pyspark.sql import functions as F

orders = spark.table(
    "dbwteam3.order_fulfillment.order_service_performance"
)

regional = spark.table(
    "dbwteam3.order_fulfillment.regional_backlog"
)

order_overdue = orders.filter(
    F.col("is_overdue") == True
).count()

regional_overdue = regional.agg(
    F.sum("overdue_orders")
).first()[0]

order_open_units = orders.agg(
    F.sum("total_open_units")
).first()[0]

regional_open_units = regional.agg(
    F.sum("total_open_units")
).first()[0]

print("Order-level overdue:", order_overdue)
print("Regional overdue sum:", regional_overdue)
print("Order-level open units:", order_open_units)
print("Regional open units:", regional_open_units)

assert order_overdue == regional_overdue, "Overdue counts do not reconcile"
assert order_open_units == regional_open_units, "Open units do not reconcile"

print("PASS: Regional backlog reconciles with order-level Gold.")


Order-level overdue: 540
Regional overdue sum: 540
Order-level open units: 57078
Regional open units: 57078
PASS: Regional backlog reconciles with order-level Gold.


In [0]:

from pyspark.sql import functions as F

schema = "dbwteam3.order_fulfillment"

lines = spark.table(f"{schema}.order_line_fulfilment")
orders = spark.table(f"{schema}.order_service_performance")
regional = spark.table(f"{schema}.regional_backlog")

checks = {
    "line_row_count": lines.count() == 4773,
    "order_row_count": orders.count() == 2040,
    "regional_row_count": regional.count() == 5,

    "duplicate_order_line_keys": (
        lines.groupBy("order_id", "line_id")
        .count().filter(F.col("count") > 1).count() == 0
    ),

    "duplicate_order_keys": (
        orders.groupBy("order_id")
        .count().filter(F.col("count") > 1).count() == 0
    ),

    "nonpositive_ordered_quantity": (
        lines.filter(F.col("ordered_quantity") <= 0).count() == 0
    ),

    "negative_open_units": (
        lines.filter(F.col("open_units") < 0).count() == 0
    ),

    "capped_delivery_exceeds_ordered": (
        lines.filter(
            F.col("capped_delivered_units") > F.col("ordered_quantity")
        ).count() == 0
    ),

    "cancelled_orders_in_otif_denominator": (
        orders.filter(
            F.col("is_cancelled")
            & F.col("otif_denominator_eligible")
        ).count() == 0
    ),

    "regional_overdue_reconciles": (
        regional.agg(F.sum("overdue_orders")).first()[0]
        == orders.filter(F.col("is_overdue")).count()
    ),

    "regional_open_units_reconcile": (
        regional.agg(F.sum("total_open_units")).first()[0]
        == orders.agg(F.sum("total_open_units")).first()[0]
    )
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

failed = [name for name, passed in checks.items() if not passed]
assert not failed, f"Gold quality checks failed: {failed}"

print(f"ALL {len(checks)} GOLD QUALITY CHECKS PASSED.")


PASS: line_row_count
PASS: order_row_count
PASS: regional_row_count
PASS: duplicate_order_line_keys
PASS: duplicate_order_keys
PASS: nonpositive_ordered_quantity
PASS: negative_open_units
PASS: capped_delivery_exceeds_ordered
PASS: cancelled_orders_in_otif_denominator
PASS: regional_overdue_reconciles
PASS: regional_open_units_reconcile
ALL 11 GOLD QUALITY CHECKS PASSED.


In [0]:

from pyspark.sql import functions as F

schema = "dbwteam3.order_fulfillment"

lines = spark.table(f"{schema}.order_line_fulfilment")
orders = spark.table(f"{schema}.order_service_performance")
regional = spark.table(f"{schema}.regional_backlog")

actual = {
    "order_line_rows": lines.count(),
    "order_rows": orders.count(),
    "regional_rows": regional.count(),
    "total_open_units": orders.agg(
        F.sum("total_open_units")
    ).first()[0],
    "otif_pass_orders": orders.agg(
        F.sum(F.col("otif_pass").cast("int"))
    ).first()[0],
    "overdue_orders": orders.filter(
        F.col("is_overdue") == True
    ).count(),
    "orders_without_lines": orders.filter(
        F.col("total_ordered_units") == 0
    ).count(),
    "regional_overdue_orders": regional.agg(
        F.sum("overdue_orders")
    ).first()[0],
    "regional_open_units": regional.agg(
        F.sum("total_open_units")
    ).first()[0]
}

expected = {
    "order_line_rows": 4773,
    "order_rows": 2040,
    "regional_rows": 5,
    "total_open_units": 57078,
    "otif_pass_orders": 1000,
    "overdue_orders": 540,
    "orders_without_lines": 40,
    "regional_overdue_orders": 540,
    "regional_open_units": 57078
}

for metric, expected_value in expected.items():
    passed = actual[metric] == expected_value
    print(
        f"{'PASS' if passed else 'FAIL'}: {metric} = "
        f"{actual[metric]} (expected {expected_value})"
    )

failed = [
    metric for metric in expected
    if actual[metric] != expected[metric]
]

assert not failed, f"Post-rerun validation failed: {failed}"

print("ALL POST-RERUN METRICS MATCH EXPECTED VALUES.")


PASS: order_line_rows = 4773 (expected 4773)
PASS: order_rows = 2040 (expected 2040)
PASS: regional_rows = 5 (expected 5)
PASS: total_open_units = 57078 (expected 57078)
PASS: otif_pass_orders = 1000 (expected 1000)
PASS: overdue_orders = 540 (expected 540)
PASS: orders_without_lines = 40 (expected 40)
PASS: regional_overdue_orders = 540 (expected 540)
PASS: regional_open_units = 57078 (expected 57078)
ALL POST-RERUN METRICS MATCH EXPECTED VALUES.


In [0]:

from pyspark.sql import functions as F

events = spark.table(
    "dbwteam3.order_fulfillment.silver_delivery_events"
)

event_check = events.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("event_id").alias("unique_event_ids"),
    F.sum(F.col("event_id").isNull().cast("int")).alias("null_event_ids")
)

event_check.show(truncate=False)


+----------+----------------+--------------+
|total_rows|unique_event_ids|null_event_ids|
+----------+----------------+--------------+
|6018      |6018            |0             |
+----------+----------------+--------------+



In [0]:

from pyspark.sql import functions as F

events = spark.table(
    "dbwteam3.order_fulfillment.silver_delivery_events"
)

events.groupBy("_batch_id").agg(
    F.count("*").alias("event_count"),
    F.countDistinct("event_id").alias("unique_event_ids"),
    F.min("delivery_time").alias("first_delivery_time"),
    F.max("delivery_time").alias("last_delivery_time")
).orderBy("_batch_id").show(truncate=False)


+---------+-----------+----------------+-------------------+-------------------+
|_batch_id|event_count|unique_event_ids|first_delivery_time|last_delivery_time |
+---------+-----------+----------------+-------------------+-------------------+
|batch_001|5115       |5115            |2026-07-30 23:57:37|2026-09-23 09:46:42|
|batch_002|453        |453             |2026-08-04 11:00:43|2026-09-25 19:02:32|
|batch_003|450        |450             |2026-08-04 15:05:10|2026-09-25 18:18:57|
+---------+-----------+----------------+-------------------+-------------------+



In [0]:

from pyspark.sql import functions as F

gold_tables = {
    "order_line_fulfilment":
        "dbwteam3.order_fulfillment.order_line_fulfilment",
    "order_service_performance":
        "dbwteam3.order_fulfillment.order_service_performance",
    "regional_backlog":
        "dbwteam3.order_fulfillment.regional_backlog"
}

baseline = {}

for name, table_name in gold_tables.items():
    df = spark.table(table_name)

    if name == "order_line_fulfilment":
        metrics = df.agg(
            F.count("*").alias("row_count"),
            F.sum("ordered_quantity").alias("ordered_units"),
            F.sum("capped_delivered_units").alias("capped_delivered_units"),
            F.sum("uncapped_delivered_units").alias("uncapped_delivered_units"),
            F.sum("open_units").alias("open_units")
        ).first().asDict()

    elif name == "order_service_performance":
        metrics = df.agg(
            F.count("*").alias("row_count"),
            F.sum("total_open_units").alias("open_units"),
            F.sum(F.col("otif_pass").cast("int")).alias("otif_pass_orders"),
            F.sum(F.col("is_overdue").cast("int")).alias("overdue_orders"),
            F.sum(F.col("total_ordered_units").eqNullSafe(0).cast("int")).alias("zero_order_unit_rows")
        ).first().asDict()

    else:
        metrics = df.agg(
            F.count("*").alias("row_count"),
            F.sum("total_open_units").alias("open_units"),
            F.sum("overdue_orders").alias("overdue_orders"),
            F.sum("orders_without_lines").alias("orders_without_lines")
        ).first().asDict()

    baseline[name] = metrics
    print(name, metrics)


order_line_fulfilment {'row_count': 4773, 'ordered_units': 319186, 'capped_delivered_units': 262108, 'uncapped_delivered_units': 262829, 'open_units': 57078}
order_service_performance {'row_count': 2040, 'open_units': 57078, 'otif_pass_orders': 1000, 'overdue_orders': 540, 'zero_order_unit_rows': 40}
regional_backlog {'row_count': 5, 'open_units': 57078, 'overdue_orders': 540, 'orders_without_lines': 40}


In [0]:

from pyspark.sql import functions as F

current = {
    "order_line_fulfilment":
        "dbwteam3.order_fulfillment.order_line_fulfilment",
    "order_service_performance":
        "dbwteam3.order_fulfillment.order_service_performance",
    "regional_backlog":
        "dbwteam3.order_fulfillment.regional_backlog"
}

for name, table_name in current.items():
    df = spark.table(table_name)

    if name == "order_line_fulfilment":
        metrics = df.agg(
            F.count("*").alias("row_count"),
            F.sum("ordered_quantity").alias("ordered_units"),
            F.sum("capped_delivered_units").alias("capped_delivered_units"),
            F.sum("uncapped_delivered_units").alias("uncapped_delivered_units"),
            F.sum("open_units").alias("open_units")
        ).first().asDict()

    elif name == "order_service_performance":
        metrics = df.agg(
            F.count("*").alias("row_count"),
            F.sum("total_open_units").alias("open_units"),
            F.sum(F.col("otif_pass").cast("int")).alias("otif_pass_orders"),
            F.sum(F.col("is_overdue").cast("int")).alias("overdue_orders"),
            F.sum(F.col("total_ordered_units").eqNullSafe(0).cast("int")).alias("zero_order_unit_rows")
        ).first().asDict()

    else:
        metrics = df.agg(
            F.count("*").alias("row_count"),
            F.sum("total_open_units").alias("open_units"),
            F.sum("overdue_orders").alias("overdue_orders"),
            F.sum("orders_without_lines").alias("orders_without_lines")
        ).first().asDict()

    matches = metrics == baseline[name]
    print(f"{name}: {'PASS' if matches else 'FAIL'}")
    print("  Baseline:", baseline[name])
    print("  Current: ", metrics)


order_line_fulfilment: PASS
  Baseline: {'row_count': 4773, 'ordered_units': 319186, 'capped_delivered_units': 262108, 'uncapped_delivered_units': 262829, 'open_units': 57078}
  Current:  {'row_count': 4773, 'ordered_units': 319186, 'capped_delivered_units': 262108, 'uncapped_delivered_units': 262829, 'open_units': 57078}
order_service_performance: PASS
  Baseline: {'row_count': 2040, 'open_units': 57078, 'otif_pass_orders': 1000, 'overdue_orders': 540, 'zero_order_unit_rows': 40}
  Current:  {'row_count': 2040, 'open_units': 57078, 'otif_pass_orders': 1000, 'overdue_orders': 540, 'zero_order_unit_rows': 40}
regional_backlog: PASS
  Baseline: {'row_count': 5, 'open_units': 57078, 'overdue_orders': 540, 'orders_without_lines': 40}
  Current:  {'row_count': 5, 'open_units': 57078, 'overdue_orders': 540, 'orders_without_lines': 40}


In [0]:

for name in ["corrected_gold", "fixed", "regional_backlog", "baseline"]:
    print(f"{name}: {'available' if name in globals() else 'not available'}")


corrected_gold: available
fixed: available
regional_backlog: available
baseline: available


In [0]:

from functools import reduce

comparisons = [
    (
        "order_line_fulfilment",
        corrected_gold,
        "dbwteam3.order_fulfillment.order_line_fulfilment"
    ),
    (
        "order_service_performance",
        fixed,
        "dbwteam3.order_fulfillment.order_service_performance"
    ),
    (
        "regional_backlog",
        regional_backlog,
        "dbwteam3.order_fulfillment.regional_backlog"
    )
]

for name, original_df, table_name in comparisons:
    current_df = spark.table(table_name)

    original_cols = set(original_df.columns)
    current_cols = set(current_df.columns)

    if original_cols != current_cols:
        print(f"{name}: FAIL — column mismatch")
        continue

    columns = sorted(original_cols)
    original = original_df.select(*columns)
    current = current_df.select(*columns)

    original_only = original.exceptAll(current).count()
    current_only = current.exceptAll(original).count()

    result = (
        "PASS"
        if original_only == 0 and current_only == 0
        else "FAIL"
    )

    print(
        f"{name}: {result} | "
        f"original_only={original_only}, "
        f"current_only={current_only}"
    )


order_line_fulfilment: PASS | original_only=0, current_only=0
order_service_performance: PASS | original_only=0, current_only=0
regional_backlog: PASS | original_only=0, current_only=0
